# Skill Selection

**Last updated:** 2026-04-09 12:00 PT

**Track:** Learning | **Sub-Ability:** Procedural learning

Can the model learn to select the correct tool/skill from a growing registry as context gets noisier?
Pre/post learning framework: study practice examples, then pick the right tool.

**Difficulty grid:** num_tools × similarity × 3 seeds = 27 items

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import sqlite3
import json
import time
import urllib.request
import urllib.error

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()


def extract_tool_name(response: str, valid_names: list) -> str:
    """Extract a tool name from model response using fuzzy matching."""
    text = strip_thinking(response)
    # Remove markdown formatting but NOT underscores (they're in tool names!)
    text = re.sub(r'[`*]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]

    # Try exact match first (case-insensitive)
    for line in reversed(lines):
        clean = line.strip('"\'').strip().lower()
        for name in valid_names:
            if clean == name.lower():
                return name

    # Try substring match on short lines (likely the answer line)
    for line in reversed(lines):
        if len(line) > 60:
            continue
        clean = line.strip('"\'').strip().lower()
        for name in valid_names:
            if name.lower() in clean:
                return name

    # Try substring match on any line
    for line in reversed(lines):
        clean = line.lower()
        for name in valid_names:
            if name.lower() in clean:
                return name

    return ''

In [ ]:
# Constants used in results analysis
NUM_TOOLS = [5, 15, 30]
SIMILARITIES = ['distinct', 'confusable', 'adversarial']
SEEDS = 3

In [ ]:
# === Load dataset from Kaggle ===
dataset = pd.read_csv('/kaggle/input/skill-selection-benchmark/skill_selection_dataset.csv')
print(f'Dataset: {len(dataset)} items')
print(dataset[['task_id', 'difficulty_label', 'expected', 'test_domain', 'item_desc']].to_string(index=False))

In [ ]:
DB_PATH = 'skill_selection_runs.db'
db = sqlite3.connect(DB_PATH)
db.execute('DROP TABLE IF EXISTS runs')
db.execute('''
    CREATE TABLE runs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT, model TEXT, task_id TEXT,
        num_tools INTEGER, similarity TEXT, difficulty_label TEXT,
        seed INTEGER, item_desc TEXT, test_domain TEXT,
        test_input TEXT, expected TEXT, valid_names TEXT,
        pre_raw TEXT, pre_extracted TEXT, pre_correct INTEGER,
        study_notes TEXT,
        post_raw TEXT, post_extracted TEXT, post_correct INTEGER,
        pre_time_s REAL, study_time_s REAL, post_time_s REAL
    )
''')
db.commit()
print(f'SQLite ready (fresh): {DB_PATH}')

In [ ]:
PRE_P = """You are using the Nexara Platform. You must select the correct tool for the task below.

{material}

--- TASK ---
{test_input}
--- END TASK ---

Based on the tool registry above, which tool should be used for this task?
Reply with ONLY the tool name (e.g. flux_bridge). Nothing else."""

STUDY_P = """You are learning to use the Nexara Platform tool registry.

{study_material}

Analyze these practice examples carefully:
1. For each example, explain WHY the correct tool was chosen.
2. Identify any patterns in how tool names, descriptions, and capabilities relate.
3. Note any cases where a tool's name or description is misleading — capabilities are what matter.
4. Write a quick-reference guide: for each tool, summarize what it ACTUALLY does in one line.

Write thorough notes you can refer to later when selecting tools."""

POST_P = """You previously studied the Nexara Platform tool registry and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Now select the correct tool for this task:

{material}

--- TASK ---
{test_input}
--- END TASK ---

Based on your study notes and the registry, which tool should be used?
Reply with ONLY the tool name (e.g. flux_bridge). Nothing else."""

## Evaluation

In [ ]:
# Small model focus — does distinct break at the floor?

# Filter to available models
available = list(kbench.llms.keys())

@kbench.task(name='skill_selection',
             description='Pre/post learning tool selection from growing registries')
def skill_selection(llm, material: str, study_material: str, test_input: str,
                    expected: str, num_tools: int, similarity: str,
                    difficulty_label: str, seed: int, item_desc: str,
                    test_domain: str, valid_names: str) -> dict:
    model_name = str(llm)
    ts = time.strftime('%Y-%m-%d %H:%M:%S')
    tid = f'{num_tools}tools_{similarity}_{seed}'
    names_list = json.loads(valid_names)

    # --- PRE: answer without study ---
    t0 = time.time()
    pre_raw = llm.prompt(PRE_P.format(material=material, test_input=test_input))
    pre_time = time.time() - t0
    pre_extracted = extract_tool_name(pre_raw, names_list)
    pre_correct = pre_extracted == expected

    # --- STUDY: analyze practice examples ---
    t0 = time.time()
    study_raw = llm.prompt(STUDY_P.format(study_material=study_material))
    study_time = time.time() - t0
    notes = strip_thinking(study_raw)

    # --- POST: answer with study notes ---
    t0 = time.time()
    post_raw = llm.prompt(POST_P.format(notes=notes, material=material, test_input=test_input))
    post_time = time.time() - t0
    post_extracted = extract_tool_name(post_raw, names_list)
    post_correct = post_extracted == expected

    try:
        db.execute(
            """INSERT INTO runs (timestamp,model,task_id,num_tools,similarity,difficulty_label,
               seed,item_desc,test_domain,test_input,expected,valid_names,
               pre_raw,pre_extracted,pre_correct,
               study_notes,post_raw,post_extracted,post_correct,
               pre_time_s,study_time_s,post_time_s)
               VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
            (ts,model_name,tid,int(num_tools),similarity,difficulty_label,int(seed),item_desc,
             test_domain,test_input,expected,valid_names,
             pre_raw,pre_extracted,int(pre_correct),
             notes,post_raw,post_extracted,int(post_correct),
             pre_time,study_time,post_time))
        db.commit()
    except Exception as e: print(f'  [SQLite warn] {e}')

    b,l = 'Y' if pre_correct else 'N', 'Y' if post_correct else 'N'
    print(f'[{model_name:40s}] [{difficulty_label:25s}] exp="{expected}" pre="{pre_extracted}"({b}) post="{post_extracted}"({l})')
    return {'pre_correct': pre_correct, 'post_correct': post_correct}

runs = skill_selection.evaluate(llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
                                n_jobs=1, timeout=3600, max_attempts=1)
print(f'\nDone: {len(runs.as_dataframe())} items')

## Results

In [ ]:
results = pd.read_sql('SELECT * FROM runs ORDER BY model, task_id', db)
print(f'Total runs: {len(results)}\n')

# === Per-model summary (scaling check) ===
print('SCALING CHECK — Pre-learning accuracy by model:')
print('=' * 70)
for model in models:
    sub = results[results['model'] == model]
    pre = sub['pre_correct'].mean()
    post = sub['post_correct'].mean()
    gain = post - pre
    print(f'  {str(model):45s}  pre={pre:.1%}  post={post:.1%}  gain={gain:+.1%}  (n={len(sub)})')

# === Per model × difficulty ===
print()
print('Pre-learning by model × difficulty:')
print('-' * 80)
header = f'{"Model":45s}'
diff_labels = sorted(results['difficulty_label'].unique())
for d in diff_labels:
    header += f'  {d:>12s}'
print(header)
for model in models:
    row = f'{str(model):45s}'
    for d in diff_labels:
        sub = results[(results['model'] == model) & (results['difficulty_label'] == d)]
        if len(sub) > 0:
            row += f'  {sub["pre_correct"].mean():>11.0%}'
        else:
            row += f'  {"--":>12s}'
    print(row)

# === By similarity (aggregated) ===
print()
print('By Similarity (all models):')
print('-' * 60)
for sim in ['distinct', 'confusable', 'adversarial']:
    sub = results[results['similarity'] == sim]
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {sim:20s}  pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

# === By num_tools (aggregated) ===
print()
print('By Num Tools (all models):')
print('-' * 60)
for nt in [5, 15, 30]:
    sub = results[results['num_tools'] == nt]
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {nt} tools             pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')

In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} ---')
    print(f'Item: {row["item_desc"]}')
    print(f'Test domain: {row["test_domain"]}')
    print(f'Expected: "{row["expected"]}"')
    print(f'Pre: "{row["pre_extracted"]}" ({b})  Post: "{row["post_extracted"]}" ({l})')
    print(f'Notes (first 300 chars): {row["study_notes"][:300]}')
    print('...' if len(str(row['study_notes'])) > 300 else '')

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Skill Selection: Pre vs Post-Learning')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('skill_selection_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Post-run: upload to BigQuery + Google Sheets via REST API ===
# Safe to do after benchmark — SDK can break, doesn't matter.

results_upload = pd.read_sql('SELECT * FROM runs', db)
db.close()
print(f'SQLite closed. {len(results_upload)} rows to upload.\n')

auth_token = None
gcp_project = None
try:
    import google.auth
    import google.auth.transport.requests
    creds, project = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    auth_token = creds.token
    gcp_project = project
    print(f'Authenticated. Project: {gcp_project}')
except Exception as e:
    print(f'Google auth not available: {e}')

BQ_DATASET = 'benchmark_runs'

# --- BigQuery ---
if auth_token and gcp_project:
    BQ_TABLE = DB_PATH.replace('_runs.db', '')
    try:
        import urllib.parse
        ds_url = f'https://api.kaggle.com/api/v1/bigquery/projects/{gcp_project}/datasets'
        ds_body = json.dumps({'datasetReference': {'datasetId': BQ_DATASET}, 'location': 'US'}).encode()
        req = urllib.request.Request(ds_url, data=ds_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        try: urllib.request.urlopen(req)
        except urllib.error.HTTPError as e:
            if e.code != 409: raise

        schema_fields = [{'name': c, 'type': 'STRING' if results_upload[c].dtype == 'object' else
                          'INTEGER' if 'correct' in c or c in ('seed','id','num_tools') else 'FLOAT'}
                         for c in results_upload.columns]
        create_url = f'https://bigquery.googleapis.com/bigquery/v2/projects/{gcp_project}/datasets/{BQ_DATASET}/tables'
        create_body = json.dumps({'tableReference': {'projectId': gcp_project, 'datasetId': BQ_DATASET, 'tableId': BQ_TABLE},
                                  'schema': {'fields': schema_fields}}).encode()
        req = urllib.request.Request(create_url, data=create_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        try: urllib.request.urlopen(req)
        except urllib.error.HTTPError as e:
            if e.code != 409: raise

        table_url = f'https://bigquery.googleapis.com/bigquery/v2/projects/{gcp_project}/datasets/{BQ_DATASET}/tables/{BQ_TABLE}/insertAll'
        rows_json = [{'json': {c: str(v) if pd.notna(v) else '' for c, v in row.items()}} for _, row in results_upload.iterrows()]
        insert_body = json.dumps({'rows': rows_json}).encode()
        req = urllib.request.Request(table_url, data=insert_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        urllib.request.urlopen(req)
        print(f'BigQuery: {len(results_upload)} rows -> {gcp_project}.{BQ_DATASET}.{BQ_TABLE}')
    except Exception as e:
        print(f'BigQuery failed (non-fatal): {e}')

# --- Google Sheets ---
if auth_token:
    SHEET_NAME = f'Learning Bench — {BQ_TABLE} Runs'
    try:
        import urllib.parse
        search_url = ('https://www.googleapis.com/drive/v3/files'
                      f'?q=name%3D%27{urllib.parse.quote(SHEET_NAME)}%27+and+mimeType%3D%27application/vnd.google-apps.spreadsheet%27'
                      '&fields=files(id,webViewLink)')
        req = urllib.request.Request(search_url, headers={'Authorization': f'Bearer {auth_token}'})
        resp = json.loads(urllib.request.urlopen(req).read())
        files = resp.get('files', [])
        if files:
            sid = files[0]['id']
        else:
            create_url = 'https://sheets.googleapis.com/v4/spreadsheets'
            req = urllib.request.Request(create_url, data=json.dumps({'properties': {'title': SHEET_NAME}}).encode(),
                                         method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
            resp = json.loads(urllib.request.urlopen(req).read())
            sid = resp['spreadsheetId']
            header = list(results_upload.columns)
            body = json.dumps({'values': [header]}).encode()
            req = urllib.request.Request(f'https://sheets.googleapis.com/v4/spreadsheets/{sid}/values/Sheet1!A1:append?valueInputOption=RAW',
                                         data=body, method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
            urllib.request.urlopen(req)

        data_rows = [[str(v) if pd.notna(v) else '' for v in row] for _, row in results_upload.iterrows()]
        body = json.dumps({'values': data_rows}).encode()
        req = urllib.request.Request(f'https://sheets.googleapis.com/v4/spreadsheets/{sid}/values/Sheet1!A1:append?valueInputOption=RAW',
                                     data=body, method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        urllib.request.urlopen(req)
        print(f'Sheets: {len(results_upload)} rows -> "{SHEET_NAME}"')
    except Exception as e:
        print(f'Sheets failed (non-fatal): {e}')

print(f'\nDone. SQLite: {DB_PATH}')